In [29]:
##### Calculate the transferability potential metric for each pixel in raster against capital and labor model training data

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import rasterio
from rasterio.transform import from_bounds


In [30]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import predictors data
capital_model = pd.read_csv(f"{cd}/Data/Clean/Training_data/capital.csv")
labor_model = pd.read_csv(f"{cd}/Data/Clean/Training_data/labor.csv")
raster_model = pd.read_parquet(f"{cd}/Data/Clean/Predictors/raster_matrix.parquet")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# Set file path to figure repo
fd = "/Users/carinamanitius/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/OneDrive/Dissertation/Chapter 1/Figures/transferability_metric"

In [31]:
##### SET MODEL

model = 'capital'
df = capital_model.copy()

In [32]:
#### Map columns across models and raster

# columns with different names
RASTER_TO_TRAIN = {
    'GDP_per_capita':            'GDP_pc',
    'soil_organic_carbon':       'SOC',
    'terrain_slope':             'slope',
    'pop_density':               'pop_density_people_per_km2',
    'travel_time_city':          'average_travel_time_city',
    'travel_time_port':          'average_travel_time_port',
    'economic_objective_prob':   'probability_economic_land_use_objective',
    'survival_objective_prob':   'probability_survival_land_use_objective',
    'season_length':             'growing_season_length_days',
    'country_labor_intensity':   'country_labor_intensity_jobs_per_million_USD',
    'country_capital_intensity': 'country_capital_intensity_USD_per_USD',
    'field_size_share_vlarge':   'share_vlarge_field',
    'field_size_share_large':    'share_large_field',
    'field_size_share_medium':   'share_medium_field',
    'field_size_share_small':    'share_small_field',
    'field_size_share_vsmall':   'share_vsmall_field',
}

# all columns (using model names) # leaving out child_dependency_ratio and USD_production_per_HA until outliers are fixed
FEATURES = [
    'SOC', 'pct_cropland_irrigated', 'female_share',
    'pop_density_people_per_km2', 'average_travel_time_city',
    'average_travel_time_port', 'probability_economic_land_use_objective',
    'probability_survival_land_use_objective', 'growing_season_length_days',
    'GDP_pc', 'slope',
    'cereals_share_production_USD', 'fibres_share_production_USD',
    'fruits_share_production_USD', 'oilcrops_share_production_USD',
    'pulses_share_production_USD', 'roots_tubers_share_production_USD',
    'rest_of_crops_share_production_USD', 'sugar_crops_share_production_USD',
    'vegetables_share_production_USD', 'rubber_share_production_USD',
    'ruminants_share_production_USD', 'monogastrics_share_production_USD',
    'poultry_share_production_USD', 'other_share_production_USD', 'pct_GDP_ag',
    'share_vlarge_field', 'share_large_field', 'share_medium_field',
    'share_small_field', 'share_vsmall_field', 'share_with_nightlights',
    'crop_intensity', 'country_labor_intensity_jobs_per_million_USD',
    'country_capital_intensity_USD_per_USD',
]

In [33]:
#### Prep data

# pull predictor data  
X_train = df[FEATURES].dropna()
X_pixel_raw = raster_model.rename(columns=RASTER_TO_TRAIN)[FEATURES] # ensure names match

# mask to only include pixels with full data
valid_mask = X_pixel_raw.notna().all(axis=1)

# rescale predictors to 0 = mean SD = 1 (puts on comparable scale)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_pixel_scaled = scaler.transform(X_pixel_raw[valid_mask])

In [20]:
#### Calculate dissimilarity index for each pixel (Meyer & Pebesma, 2021)

# Calculate normalization factor from training data
# (the average 'distance' from each observation to the next closest observation)
nn_train = NearestNeighbors(n_neighbors=2, n_jobs=-1) # set method to find nearest neighbor (2nd because 1st is itself)
nn_train.fit(X_train_scaled) # set reference data 

train_nn_dists, _ = nn_train.kneighbors(X_train_scaled) # apply function to find nearest neighboor 
avg_train_nn_dist = train_nn_dists[:, 1].mean() # calculate average across training data, keeping only data for 2nd neighbor (not self)


# Calculate dissimilarity index for each pixel
nn = NearestNeighbors(n_neighbors=1, n_jobs=-1) # set method to find nearest neighbor in training data (1st because self isn't in training data)
nn.fit(X_train_scaled) # set reference data 

pixel_nn_dists, _ = nn.kneighbors(X_pixel_scaled) # apply function to find nearest neighboor for each pixel
pixel_nn_dists = pixel_nn_dists[:, 0] # convert to vector

DI = pixel_nn_dists / avg_train_nn_dist # calculate DI by normalizing against avg_train_nn_dist


# Calculate area of applicability threshold (Meyer & Pebesma, 2021)
# threshold is 95th percentile of DI distribution (from training points)
train_di = train_nn_dists[:, 1] / avg_train_nn_dist  # calculate DI for training data
aoa_threshold = np.percentile(train_di, 95) # find the 95th percentile 

In [21]:
### Assemble output

# get x,y of pixel  
output_df = raster_model[['x', 'y']].copy()

# initialize columns 
output_df['DI'] = np.nan
output_df['within_AOA'] = np.nan

# add in DI and binary measure of within AOA for each pixel 
output_df.loc[valid_mask, 'DI'] = DI
output_df.loc[valid_mask, 'within_AOA'] = (DI <= aoa_threshold).astype(int)

output_df.to_csv(f"{fd}/DI_AOA_calcs_{model}.csv", index=False)

In [22]:
### Convert back to raster to plot 

# get metadata from raster
with rasterio.open(ref_path) as src:
    profile = src.profile
    height  = src.height
    width   = src.width
    transform = src.transform

# start with empty aray in correct format
di_grid  = np.full((height, width), np.nan, dtype=np.float32)
aoa_grid = np.full((height, width), np.nan, dtype=np.float32)

# map x/y coordinates back to arrary using row/column indicies 
rows, cols = rasterio.transform.rowcol(
    transform,
    output_df['x'].values,
    output_df['y'].values
)
rows = np.array(rows)
cols = np.array(cols)

# fill with DI and AOA values (only for pixels that aren't missing values)
valid = output_df['DI'].notna().values
di_grid[rows[valid], cols[valid]]  = output_df.loc[valid, 'DI'].values
aoa_grid[rows[valid], cols[valid]] = output_df.loc[valid, 'within_AOA'].values

# Save as tiffs
profile.update(dtype='float32', count=1, nodata=np.nan)

with rasterio.open(f'{fd}/DI_{model}.tif', 'w', **profile) as dst:
    dst.write(di_grid, 1)

with rasterio.open(f'{fd}/AOA_{model}.tif', 'w', **profile) as dst:
    dst.write(aoa_grid, 1)